# Code Review Assistant: full run on Kaggle

This notebook runs my code review tool end to end. The rule engine finds problems and points to the exact line, then a local code model (Qwen2.5-Coder) explains each one, suggests a fix and looks for logic bugs the rules can't catch. It also scores two model sizes against each other, checks the tool on real code, and runs the slow parts in parallel.

**Why Kaggle and not Colab.** The brief asks for a Colab notebook, but I ran this on Kaggle instead. Kaggle's free tier gives two T4 GPUs at once, so I could load one copy of the model on each GPU and score both models in about half the time, while free Colab gives one. Kaggle's *Save & Run All* also runs the whole notebook on their servers as a saved version, so the run finishes even if my browser closes, and the log and output files stay with that version. The notebook still works on Colab without changes: it detects where it is running and clones the code from GitHub when there is no Kaggle dataset.

**Before running** (right-hand panel):

1. Accelerator: **GPU T4 x2**
2. Internet: **On**
3. Add Input: upload `code_review_assistant.py` as a dataset (and use its newest version)
4. **Save Version → Save & Run All**. Takes about 15 minutes. Results end up in the Output tab.

**What runs when**

| Lane | What | Uses | When |
| --- | --- | --- | --- |
| A | Download both models | network | first, in the background |
| B | Self-test, scoring without a model, stdlib benchmark, ruff/bandit check, audit sample | CPU | while A downloads |
| C | Smoke test, then score the 3B and 7B models, test cases split across both GPUs | GPU | after A and B |
| D | Rating sheet, batch review of real files, UI check | GPU + CPU | last |

## 0. Settings

This is the only cell you should need to change.

In [ ]:
REPO_URL = "https://github.com/hansineesoundararajan/code-review-assistant.git"

LLM = "hf"                                   # "hf" runs a local model. "offline" dry-runs the plumbing with no model.
MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"     # about 6 GB in fp16, fits one T4 with room to spare

RUN_7B = True                                # also score the 7B model in 4-bit, for the model-selection comparison
MODEL_7B = "Qwen/Qwen2.5-Coder-7B-Instruct"
RUN_BATCH_DEMO = True                        # review a few real stdlib files with the model
RUN_UI_LIVE = False                          # True only in an interactive session: the live UI never returns

## 1. Environment

Checks for GPUs and internet, and sets up a small helper that prints command output straight into the cell.

In [ ]:
import csv
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

# Keep the log readable: no per-tensor loading bars, no repeated library notices.
# Real errors still print.
os.environ.update({
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "HF_HUB_VERBOSITY": "error",
    "TRANSFORMERS_VERBOSITY": "error",
    "TOKENIZERS_PARALLELISM": "false",
})

T0 = time.time()
TIMELINE = []


def sh(cmd, env=None):
    """Run a shell command and stream its output into the cell.

    Output from a plain os.system call can end up in the server log instead of
    the cell when a notebook runs headless, so the pipe is read here.
    """
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, env={**os.environ, **(env or {})})
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


def stamp(label):
    TIMELINE.append((time.time() - T0, label))
    print(f"[{time.time() - T0:5.0f}s] {label}")


def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=5)
        return True
    except Exception:
        return False


ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()
os.chdir(WORK)

CORES = os.cpu_count() or 1
smi = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
GPUS = smi.stdout.strip().splitlines() if smi.returncode == 0 else []
N_GPU = max(len(GPUS), 1) if LLM == "hf" else 1
ONLINE = reachable("https://huggingface.co")

print("platform :", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local")
print("python   :", sys.version.split()[0])
print("cpu cores:", CORES)
print("gpus     :", len(GPUS), GPUS)
print("internet :", ONLINE)

LLM_READY = LLM != "hf" or (len(GPUS) > 0 and ONLINE)
if not LLM_READY:
    print("\nThe model lanes will be skipped.")
    if not GPUS:
        print("  no GPU: set Accelerator to GPU T4 x2 in the side panel")
    if not ONLINE:
        print("  no internet: switch Internet on in the side panel")
stamp("environment checked")

## 2. Get the code

The whole tool is one file. The notebook looks for it in the uploaded Kaggle dataset first, then clones it from GitHub. It also checks the version, because Kaggle keeps using the dataset version you first attached until you switch it, and that caught me out twice.

In [ ]:
EXPECTED_VERSION = "1.2.0"     # must match __version__ in the code_review_assistant.py you uploaded


def find_source():
    uploaded = sorted(Path("/kaggle/input").glob("**/code_review_assistant*.py"),
                      key=lambda p: p.stat().st_mtime, reverse=True)
    if uploaded:
        return uploaded[0]                  # newest upload wins if there are several
    here = WORK / "code_review_assistant.py"
    return here if here.exists() else None


source = find_source()
if source is None:
    if "<your-username>" in REPO_URL:
        raise RuntimeError("Set REPO_URL in the settings cell, or upload code_review_assistant.py "
                           "as a dataset (Add Input, then Upload).")
    if sh(f"git clone -q {REPO_URL} repo") != 0:
        raise RuntimeError(f"could not clone {REPO_URL}")
    source = WORK / "repo" / "code_review_assistant.py"

target = WORK / "code_review_assistant.py"
if source.resolve() != target.resolve():
    shutil.copy(source, target)             # always a clean name, so imports and the shell both work
print("using", source)

CRA = f"{sys.executable} code_review_assistant.py"
found_version = subprocess.run(f"{CRA} --version", shell=True, capture_output=True,
                               text=True).stdout.split()[-1]
print("code_review_assistant", found_version)
if found_version != EXPECTED_VERSION:
    # Kaggle pins each input to the dataset version that was current when it was
    # added, so uploading a new version does not reach the notebook by itself.
    raise RuntimeError(
        f"This is version {found_version} of code_review_assistant.py, but the notebook expects "
        f"{EXPECTED_VERSION}. Kaggle is still using an older upload. In the right-hand panel, open "
        "Input, find the dataset, and choose the newest version (the three-dot menu, or the "
        "'new version available' notice), then run again.")

## 3. Tools

- `ruff` does the job of flake8, pyflakes, bugbear and a lot of bandit in one fast call (about 10 ms a file).
- `bandit` adds its own security checks.
- `gradio` is only needed for the UI check at the end.

In [ ]:
sh(f"{sys.executable} -m pip install -q ruff bandit gradio")

missing = [m for m in ("transformers", "accelerate", "bitsandbytes") if importlib.util.find_spec(m) is None]
if missing:
    sh(f"{sys.executable} -m pip install -q {' '.join(missing)}")

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"   # faster model downloads
print("ruff  :", shutil.which("ruff"))
print("bandit:", shutil.which("bandit"))
stamp("tools installed")

## 4. Lane A: start the model downloads

This is the slowest step and doesn't need the CPU, so it starts first and runs in the background.

In [ ]:
downloads = {}

if LLM_READY and LLM == "hf":
    log = open("download.log", "w")

    def start_download(repo_id):
        code = f"from huggingface_hub import snapshot_download; print(snapshot_download({repo_id!r}))"
        downloads[repo_id] = subprocess.Popen([sys.executable, "-c", code],
                                              stdout=log, stderr=subprocess.STDOUT)

    start_download(MODEL)
    if RUN_7B:
        start_download(MODEL_7B)
    stamp("lane A: model downloads running in the background")
else:
    print("lane A skipped")

## 5. Lane B: CPU work while the models download

The test set has 40 small files. 30 have problems the rules are built to catch, plus some clean files and edge cases (empty file, only comments, a syntax error). The other 10 have logic bugs no rule can see, like a loop that skips the first item or `>` where `>=` was meant. Without a model the score on those 10 should be zero, so anything found there is down to the model.

Each bug is labelled with a `#@ bug` comment on its line. Those comments are removed before review, so neither the rules nor the model can see the answers.

In [ ]:
sh(f"{CRA} selftest | grep -E 'FAIL|checks passed'")
sh(f"{CRA} dump-corpus samples")
sh(f"{CRA} eval -o eval_static.json")
stamp("lane B: self-test and static evaluation done")

### Benchmark on real code: one core vs all of them

Runs the rules over the whole Python standard library, once with one worker and once with every core.

In [ ]:
timings = {}
for workers in sorted({1, CORES}):
    sh(f"{CRA} benchmark --workers {workers} -o bench_{workers}w.json > /dev/null")
    timings[workers] = json.load(open(f"bench_{workers}w.json"))

print(f"{'workers':>8} {'wall (s)':>9} {'lines/s':>10} {'findings':>9}")
for w, data in timings.items():
    print(f"{w:>8} {data['wall_seconds']:>9.1f} {data['lines_per_second']:>10,.0f} {data['total_findings']:>9,}")
if CORES > 1:
    print(f"\nspeedup with {CORES} workers: "
          f"{timings[1]['wall_seconds'] / timings[CORES]['wall_seconds']:.2f}x, same findings either way")

shutil.copy(f"bench_{CORES}w.json", "benchmark_results.json")
data = timings[CORES]
print(f"\n{data['files_scanned']} files, {data['total_lines']:,} lines: "
      f"{data['findings_per_100_lines']} findings per 100 lines, "
      f"{data['serious_per_100_lines']} critical or high per 100 lines")
stamp("lane B: stdlib benchmark done")

### Ruff and bandit

Quick check that both tools are installed and that their output still parses. Anything they report that one of my rules already covers is dropped, so nothing shows up twice.

In [ ]:
sys.path.insert(0, str(WORK))
from code_review_assistant import ExternalTools, SourceUnit

probe = SourceUnit("import os\nimport sys\n\n\ndef f(x=[]):\n    eval(x)\n", "probe.py")
for tool in ("ruff", "bandit"):
    if not shutil.which(tool):
        print(f"{tool}: not installed, skipped")
        continue
    found = ExternalTools(enabled=True, tools=(tool,)).run(probe)
    print(f"{tool}: {len(found)} findings -> {[f.rule for f in found][:6]}")
    if not found:
        print(f"  WARNING: {tool} is installed but returned nothing; its output format may have changed")

if shutil.which("ruff") or shutil.which("bandit"):
    sh(f"{CRA} review samples --external-tools --format json -o review_tools.json")
    data = json.load(open("review_tools.json"))
    rules = [f["rule"] for item in data["files"] for f in item["findings"]]
    print(f"\ncorpus review with the tools on: {len(rules)} findings, "
          f"{sum(r.startswith('ruff:') for r in rules)} ruff-only, "
          f"{sum(r.startswith('bandit:') for r in rules)} bandit-only "
          "(anything our rules already cover is dropped, not shown twice)")

### Precision check on code I didn't write

A perfect score on test files I wrote myself doesn't prove much. So this picks 50 findings from the standard library and saves them to `audit_sheet.csv`, where each one gets marked **real**, **false** or **unsure** by hand. The share marked real is the precision on outside code.

The first sample I checked (seed 7) came out at 26%. Three rules caused most of the false alarms (unused variables in tuple unpacking, `in` followed by indexing, and `assert` on internal state), so I narrowed them. This run uses a different sample (seed 11), so the new number isn't measured on the same rows I used to fix the rules.

In [ ]:
import random
import sysconfig

stdlib = Path(sysconfig.get_paths()["stdlib"])
pool = [p for p in sorted(stdlib.rglob("*.py"))
        if not {"test", "tests", "idlelib", "site-packages"} & set(p.parts)]
random.seed(11)   # a fresh sample: seed 7's rows were used to tune three rules
picked = random.sample(pool, min(80, len(pool)))

sh(f"{CRA} review {' '.join(str(p) for p in picked)} --format json -o audit_review.json > /dev/null")
data = json.load(open("audit_review.json"))

by_category = {}
for item in data["files"]:
    for f in item["findings"]:
        by_category.setdefault(f["category"], []).append((item["path"], f))
for rows in by_category.values():
    random.shuffle(rows)

sample = []
while len(sample) < 50 and any(by_category.values()):
    for rows in by_category.values():            # take one from each category in turn
        if rows and len(sample) < 50:
            sample.append(rows.pop())


def context(path, line, radius=2):
    lines = Path(path).read_text(errors="replace").splitlines()
    low, high = max(line - 1 - radius, 0), min(line + radius, len(lines))
    return "\n".join(f"{n + 1:>4}  {lines[n]}" for n in range(low, high))


with open("audit_sheet.csv", "w", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["file", "line", "category", "rule", "severity", "issue", "code", "verdict", "comment"])
    for path, f in sorted(sample, key=lambda pair: (pair[1]["category"], pair[0], pair[1]["line"])):
        writer.writerow([Path(path).name, f["line"], f["category"], f["rule"], f["severity"],
                         f["message"], context(path, f["line"]), "", ""])

counts = {c: sum(f["category"] == c for _, f in sample) for c in by_category}
print(f"{len(picked)} files reviewed, {sum(len(i['findings']) for i in data['files'])} findings in total")
print(f"audit_sheet.csv: {len(sample)} findings sampled, by category: {counts}")
stamp("lane B: audit sheet written")

## 6. Wait for the models

Prints the download progress until both models are on disk.

In [ ]:
def cache_gb(repo_id):
    try:
        from huggingface_hub.constants import HF_HUB_CACHE
    except ImportError:
        HF_HUB_CACHE = str(Path.home() / ".cache/huggingface/hub")
    folder = Path(HF_HUB_CACHE) / ("models--" + repo_id.replace("/", "--"))
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) / 1e9


deadline = time.time() + 3600
while any(p.poll() is None for p in downloads.values()):
    if time.time() > deadline:
        raise TimeoutError("model download is taking over an hour; check download.log")
    sizes = ", ".join(f"{m.split('/')[-1]} {cache_gb(m):.1f} GB" for m, p in downloads.items() if p.poll() is None)
    print(f"[{time.time() - T0:5.0f}s] downloading: {sizes}")
    time.sleep(15)

failed = [m for m, p in downloads.items() if p.returncode != 0]
if failed:
    print(open("download.log").read()[-2000:])
    raise RuntimeError(f"download failed for {failed}")
if downloads:
    os.environ["HF_HUB_OFFLINE"] = "1"      # everything is cached; no more network calls
stamp("lane A: model files are on disk")

## 7. Lane C: the models

### Smoke test

Two files on one GPU. If the model's replies can't be read as JSON, it shows up here before the long runs.

In [ ]:
if LLM_READY:
    sh(f"{CRA} review samples/sql_injection.py samples/off_by_one_sum.py --llm {LLM} --model {MODEL} "
       "--workers 1 --format json -o smoke.json", env={"CUDA_VISIBLE_DEVICES": "0"})
    data = json.load(open("smoke.json"))
    findings = [f for item in data["files"] for f in item["findings"]]
    explained = sum(f["source"] == "static+llm" for f in findings)
    own = [f for f in findings if f["source"] == "llm"]
    print(f"\nexplained {explained} static findings, proposed {len(own)} of its own")
    for f in own:
        print(f"  line {f['line']}: {f['message']} (confidence {f['confidence']})")
    for item in data["files"]:
        for e in item["errors"]:
            print("  note:", e)
else:
    print("skipped")

### Score the 3B model

The test cases are split across both GPUs, with one copy of the model on each. The timings per file don't include loading the model.

In [ ]:
if LLM_READY:
    sh(f"{CRA} eval --llm {LLM} --model {MODEL} --workers {N_GPU} --gpus auto --show-errors -o eval_llm.json")
    stamp("lane C: 3B evaluation done")
else:
    print("skipped")

### Score the 7B model (4-bit)

Loaded in 4-bit, so it takes about 5 GB and fits on one T4. This is the comparison for picking a model.

In [ ]:
if LLM_READY and RUN_7B and LLM == "hf":
    sh(f"{CRA} eval --llm hf --model {MODEL_7B} --quantize 4bit --workers {N_GPU} --gpus auto "
       "--show-errors -o eval_llm_7b.json")
    stamp("lane C: 7B evaluation done")
else:
    print("skipped")

### Rules only vs each model

The table for the report. The rows that matter most:

- **Rule-blind recall**: how many of the 10 logic bugs the model found.
- **Own findings precision**: how often the model's own findings were right.
- **Specific explanations**: how often the explanation names something in the actual code.

On the 10 logic-bug cases a finding only counts if it's on the right line *and* its explanation describes the actual bug (each case has a few keywords). Right line, wrong reason counts as wrong.

In [ ]:
from code_review_assistant import format_eval_markdown

runs = {"static only": "eval_static.json", "3B": "eval_llm.json", "7B 4-bit": "eval_llm_7b.json"}
results = {name: json.load(open(path)) for name, path in runs.items() if Path(path).exists()}


def own_precision(r):
    added = r["llm"]["added"]
    return f"{(added - r['llm']['added_wrong']) / added:.3f}" if added else "-"


def specific_share(r):
    n = r["llm"]["enriched"]
    return f"{r['llm']['specific'] / n:.0%}" if n else "-"


rows = [
    ("overall F1", lambda r: f"{r['overall']['f1']:.3f}"),
    ("rule-visible F1", lambda r: f"{r['subsets']['static_visible']['f1']:.3f}"),
    ("rule-blind recall", lambda r: f"{r['subsets']['static_blind']['recall']:.3f}"),
    ("own findings kept", lambda r: str(r["llm"]["added"])),
    ("own findings precision", own_precision),
    ("static findings explained", lambda r: f"{r['llm']['enriched']}/{r['llm']['static_findings']}"),
    ("specific explanations", specific_share),
    ("mean latency (ms)", lambda r: f"{r['latency']['mean_seconds'] * 1000:,.1f}"),
    ("wall clock (s)", lambda r: f"{r['latency']['wall_seconds']:.0f}"),
]

names = list(results)
print(f"{'':<27}" + "".join(f"{n:>14}" for n in names))
print("-" * (27 + 14 * len(names)))
for label, get in rows:
    print(f"{label:<27}" + "".join(f"{get(results[n]):>14}" for n in names))

table = ["| | " + " | ".join(names) + " |", "| --- |" + " ---: |" * len(names)]
for label, get in rows:
    table.append(f"| {label} | " + " | ".join(get(results[n]) for n in names) + " |")
with open("model_comparison.md", "w") as fh:
    fh.write("\n".join(table) + "\n")
    for name in names[1:]:
        fh.write(f"\n### {name}\n\n{format_eval_markdown(results[name])}\n")
print("\nwrote model_comparison.md (tables ready for REPORT.md)")

### Rating sheet

The brief asks for the optimisation suggestions to be rated by reviewers. A program can't do that fairly, so this saves 20 of the 7B model's explanations and fixes (optimisation ones first) to `rating_sheet.csv`, with an empty column to score each one from 1 to 5.

In [ ]:
# The examples people rate should come from the model the comparison recommends.
if RUN_7B and LLM == "hf":
    REVIEW_ARGS, REVIEW_NAME = f"--llm hf --model {MODEL_7B} --quantize 4bit", "7B 4-bit"
else:
    REVIEW_ARGS, REVIEW_NAME = f"--llm {LLM} --model {MODEL}", MODEL.split("/")[-1]

if LLM_READY:
    sh(f"{CRA} review samples {REVIEW_ARGS} --workers {N_GPU} --gpus auto "
       "--format json -o review_llm_samples.json")
    data = json.load(open("review_llm_samples.json"))
    rated = [(item["path"], f) for item in data["files"] for f in item["findings"]
             if f["source"] in ("static+llm", "llm")]
    rated.sort(key=lambda pair: (pair[1]["category"] != "optimization", pair[0], pair[1]["line"]))
    rated = rated[:20]

    with open("rating_sheet.csv", "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["file", "line", "category", "rule", "issue", "explanation", "fix",
                         "usefulness_1_to_5", "comment"])
        for path, f in rated:
            writer.writerow([Path(path).name, f["line"], f["category"], f["rule"], f["message"],
                             f["rationale"], f["suggestion"], "", ""])
    print(f"wrote rating_sheet.csv with {len(rated)} rows from the {REVIEW_NAME} model "
          f"({sum(f['category'] == 'optimization' for _, f in rated)} optimisation suggestions)")
    stamp("lane D: rating sheet written")
else:
    print("skipped")

### Batch mode on real files

Reviews a few files from the standard library's `json` package with the 7B model, two files at a time (one per GPU).

In [ ]:
if LLM_READY and RUN_BATCH_DEMO:
    import sysconfig

    stdlib = Path(sysconfig.get_paths()["stdlib"])
    files = " ".join(str(p) for p in sorted((stdlib / "json").glob("*.py"))[:6])

    start = time.time()
    sh(f"{CRA} review {files} {REVIEW_ARGS} --workers {N_GPU} --gpus auto "
       "--min-severity medium --format markdown -o batch_review.md")
    text = open("batch_review.md").read()
    print(f"\n{time.time() - start:.0f}s wall clock with {N_GPU} worker(s), {REVIEW_NAME} model")
    print("model output that could not be parsed:", text.count("Could not parse model output"), "chunk(s)")
    print("\n" + "".join(text.splitlines(True)[:30]))
    stamp("lane D: batch demo done")
else:
    print("skipped")

## 8. Interactive UI

Builds the Gradio UI and calls its review function directly, which tests the same path the button uses. Then it starts the server for a few seconds to check it actually serves. Set `RUN_UI_LIVE = True` in an interactive session to keep it open with a public link.

In [ ]:
if importlib.util.find_spec("gradio") is None:
    print("gradio not installed, UI check skipped")
else:
    from code_review_assistant import EXAMPLE_CODE, build_ui

    try:
        demo, review_fn = build_ui()                 # static backend: fast and needs no GPU
        summary, metrics, report_md = review_fn(EXAMPLE_CODE, ["bug", "security", "style", "optimization"], "info")
        print("UI review function:", summary.splitlines()[0])

        demo.launch(prevent_thread_lock=True, share=False, quiet=True)
        time.sleep(3)
        print("UI served at:", demo.local_url)
        demo.close()
        stamp("UI checked")
    except Exception as exc:
        # reported, not raised, so a gradio version quirk cannot stop the run before the summary
        print(f"UI check FAILED: {type(exc).__name__}: {exc}")

    if RUN_UI_LIVE:
        sh(f"{CRA} ui --share --llm {LLM} --model {MODEL}")

## 9. Summary

How long each step took, and which files were written. Download them from the version's Output tab.

In [ ]:
stamp("finished")
print("\ntimeline")
for seconds, label in TIMELINE:
    print(f"  {seconds:6.0f}s  {label}")

print("\nfiles written to", WORK)
for pattern in ("*.json", "*.md", "*.csv", "*.log"):
    for p in sorted(WORK.glob(pattern)):
        print(f"  {p.name:<28}{p.stat().st_size:>10,} bytes")
if ON_KAGGLE:
    print("\nAfter Save & Run All, download these from the version's Output tab.")